# QuantProbe — Phase 2 extraction (Colab)

Runs Stage A at **FP16, INT8 and INT4** and caches the hidden states to Google Drive.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU`. Nothing here works on CPU —
   `bitsandbytes` is CUDA-only.
2. If you are running the **Llama** model: have your HF token ready and make sure
   the licence at `huggingface.co/meta-llama/Llama-3.2-1B-Instruct` is **granted**,
   not `PENDING`.
3. If Llama access is still pending, use the **Qwen** config in step 6 instead —
   it is ungated, needs no token, and exercises the identical code path.

Run the cells top to bottom. ~20–50 min, mostly waiting.

---

## 1 — Check the GPU

If this prints `CUDA available: False`, stop: `Runtime → Change runtime type → T4 GPU`,
then `Runtime → Restart session`, then run this again.

In [ ]:
!nvidia-smi

import torch
print()
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1024**3))
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')

## 2 — Install pinned dependencies

Only the four libraries whose versions can move a number. Colab's own CUDA build of
`torch` is deliberately left alone — installing our CPU pin would replace it and
break the GPU.

Red text about dependency conflicts is Colab complaining about its own preinstalled
packages. Expected. Ignore it.

In [ ]:
!pip install -q -U transformers==5.16.1 datasets==5.0.1 accelerate==1.14.0 bitsandbytes==0.50.2

import importlib
for m in ('transformers', 'datasets', 'accelerate', 'bitsandbytes'):
    mod = importlib.import_module(m)
    print(f'{m:15s} {getattr(mod, "__version__", "?")}')

## 3 — Get the repo onto Colab

Upload `quantprobe_repo.zip` when the `Choose Files` button appears.

In [ ]:
from google.colab import files
import zipfile, os, shutil

# Step OUT before deleting. On a re-run this process is already sitting
# inside /content/QuantProbe (we chdir there at the end of this cell), so
# rmtree would leave cwd pointing at a deleted inode. files.upload() then
# fails with FileNotFoundError while saving the file it just received -
# the upload succeeds, the write does not.
os.chdir('/content')
shutil.rmtree('/content/QuantProbe', ignore_errors=True)

print('Select quantprobe_repo.zip ...')
uploaded = files.upload()
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall('/content/QuantProbe')

os.chdir('/content/QuantProbe')
print()
print('repo at', os.getcwd())
print(sorted(os.listdir('.')))

assert os.path.isfile('scripts/01_extract.py'), 'zip did not contain the repo'
print()
print('OK - now re-run steps 4, 5 and 6 before the smoke test.')

## 4 — Mount Drive and point the cache at it

This is the cell that makes a disconnect survivable. Written to Drive the cache
persists; written to `/content` it evaporates when the runtime recycles.

`QUANTPROBE_CACHE` is the *root* — `datasets/` and `activations/` go underneath it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
CACHE = '/content/drive/MyDrive/quantprobe_cache'
os.makedirs(CACHE, exist_ok=True)

os.environ['QUANTPROBE_CACHE'] = CACHE
os.environ['HF_HOME'] = '/content/hf'              # weights are re-downloadable; keep off Drive
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # must be set before the first CUDA call

print('QUANTPROBE_CACHE = %s' % CACHE)
print('free on Drive    = %.1f GB   (need ~3 GB)' % (shutil.disk_usage(CACHE).free/1024**3))

## 5 — Pick the model

**Change this one line and nothing else.**

| Config | Model | Gated? |
|---|---|---|
| `configs/model_llama1b.yaml` | Llama-3.2-1B-Instruct | **yes** — needs token + granted licence |
| `configs/model_qwen15b.yaml` | Qwen2.5-1.5B-Instruct | no — runs immediately |

If your Llama request still says `PENDING`, use the Qwen line. It exercises the
identical code path, so everything you learn transfers.

In [ ]:
MODEL_CONFIG = 'configs/model_llama1b.yaml'
# MODEL_CONFIG = 'configs/model_qwen15b.yaml'   # <- ungated, use while Llama is PENDING

import os, sys, yaml
os.environ['MODEL_CONFIG'] = MODEL_CONFIG

with open(MODEL_CONFIG) as f:
    _cfg = yaml.safe_load(f)
MODEL_SHORT = _cfg['model']['short_name']
os.environ['MODEL_SHORT'] = MODEL_SHORT
GATED = bool(_cfg['model'].get('gated', False))

print('config      :', MODEL_CONFIG)
print('model       :', _cfg['model']['hf_id'])
print('short name  :', MODEL_SHORT)
print('gated       :', GATED, '-> token required' if GATED else '-> no token needed')

## 6 — HuggingFace token

Only needed for the gated (Llama) config; skipped automatically otherwise.

`getpass` so the token is typed, not stored. **Never paste a token into a cell** —
a saved notebook containing one is a leaked credential, and notebooks get shared
by accident constantly.

In [ ]:
import getpass, os, subprocess, sys

if not GATED:
    print('Ungated model selected - no token needed. Skipping.')
else:
    tok = getpass.getpass('HF token (input hidden): ').strip()
    os.environ['HF_TOKEN'] = tok

    # Belt AND braces. The extraction runs as `!python ...`, which is a
    # SEPARATE process. Whether os.environ crosses that boundary is one
    # more thing that can silently fail. login() writes the token to
    # ~/.cache/huggingface/token, a file every process reads regardless.
    from huggingface_hub import login, HfApi
    login(token=tok, add_to_git_credential=False)
    print('token set, %d chars, and written to the hub credential file' % len(tok))

    # 1. can THIS process reach the model?
    try:
        HfApi().model_info(_cfg['model']['hf_id'], token=tok)
        print('notebook  -> access to %s: OK' % _cfg['model']['hf_id'])
    except Exception as e:
        print('notebook  -> ACCESS FAILED:', type(e).__name__)
        print(str(e)[:300])
        print('Check https://huggingface.co/settings/gated-repos says ACCEPTED.')

    # 2. can a SUBPROCESS reach it? This is what actually runs the job,
    #    and it is the step that was silently failing before.
    probe = (
        'from huggingface_hub import get_token, HfApi; '
        't = get_token(); '
        "print('subprocess -> token visible:', bool(t)); "
        "HfApi().model_info('" + _cfg['model']['hf_id'] + "', token=t); "
        "print('subprocess -> access to the model: OK')"
    )
    r = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])

    print()
    print('Both lines must say OK before you run the smoke test.')

## 7 — Smoke test  ← THE IMPORTANT ONE

Ten examples through all three precisions. 2–4 minutes including the download.

**Read this output, do not skim it.** You are looking for, on the INT8 and INT4 arms:

```
int4 quantization confirmed: NNN quantized (uint8) parameter tensors
param dtype counts: {'torch.uint8': NNN, 'torch.float16': NN}
```

If bitsandbytes silently fails to quantize, `from_pretrained` still returns a
perfectly working FP16 model, every downstream number looks sane, and the drift
curve sits at zero — the conclusion "quantization doesn't hurt the probe" would be
pure artefact. The script now hard-fails on that, but seeing the confirmation is how
you know the study is real.

**Do not run step 8 until this prints `PHASE 2 EXTRACTION COMPLETE`.**

In [ ]:
!python scripts/01_extract.py --config $MODEL_CONFIG --limit 10 --force

## 8 — Full extraction

All 6049 statements × 3 precisions. **15–40 minutes.** Progress prints as it goes.

Keep the tab open and the laptop awake — Colab kills idle sessions.

**If it disconnects: rerun this same cell.** It resumes from the last flushed batch
rather than starting over. If the session fully restarted, rerun steps 4 and 5 first.

In [ ]:
!python scripts/01_extract.py --config $MODEL_CONFIG --resume

## 9 — Verify the done-signal

Checks all three arms saw byte-identical tokens **and** were extracted with identical
settings. `batch_size` is in that check for a measured reason: different padding
widths change matmul reduction order and shift activations by ~3e-4, which is pure
artefact sitting inside exactly what E3 measures.

In [ ]:
!python scripts/01_extract.py --config $MODEL_CONFIG --verify-only

## 10 — Look at what came out

**The one thing to check: layer 0 must be ~0.00000 in both columns.**

Layer 0 is the embedding output, and bitsandbytes does not quantize embeddings — so
it *cannot* have drifted. If it did, extraction is broken, not the science.

After that, drift should grow with depth as rounding errors compound.

In [ ]:
import sys, os, numpy as np
sys.path.insert(0, '/content/QuantProbe')
from pathlib import Path
from src.extract.hidden import load_extraction, extraction_dir

root = Path(os.environ['QUANTPROBE_CACHE'])
arms = {p: load_extraction(extraction_dir(root, MODEL_SHORT, p))
        for p in ('fp16', 'int8', 'int4')}

for p, d in arms.items():
    h = d['hidden']
    print(f"{p:5s} shape={h.shape} dtype={h.dtype} positive_rate={d['labels'].mean():.3f}")

S = 500
fp16 = np.asarray(arms['fp16']['hidden'][:S])
i8   = np.asarray(arms['int8']['hidden'][:S])
i4   = np.asarray(arms['int4']['hidden'][:S])

print()
print('mean L2 distance from FP16, per layer (%d rows):' % S)
print(f"{'layer':>6} {'INT8':>12} {'INT4':>12}")
for L in range(fp16.shape[1]):
    d8 = np.linalg.norm(i8[:, L] - fp16[:, L], axis=1).mean()
    d4 = np.linalg.norm(i4[:, L] - fp16[:, L], axis=1).mean()
    print(f'{L:6d} {d8:12.5f} {d4:12.5f}')

print()
print('layer 0 must be ~0 - embeddings are not quantized.')

## 11 — Confirm the cache is safe on Drive

Once these files exist, the expensive part is done and survives a disconnect.
Phases 3–5 read only these arrays and need no GPU.

In [ ]:
!du -sh $QUANTPROBE_CACHE/activations/$MODEL_SHORT/* 2>/dev/null
!find $QUANTPROBE_CACHE/activations -name 'hidden.npy' -exec ls -lh {} \;

print()
print('Copy the output of steps 7, 9, 10 and 11 back into the chat.')
print('To bring the arrays down: Drive -> quantprobe_cache -> right-click')
print('activations -> Download, then unzip into D:\\quantprobe_cache\\')